In [ ]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt

# تنظیمات Tile Coding (مشابه سوال قبل برای حفظ ثبات)
num_tilings = 8
tiles_per_dim = 8
num_actions = 3  # 0: Left, 1: Idle, 2: Right

# محدوده حالت‌ها در Gymnasium
pos_min, pos_max = -1.2, 0.6
vel_min, vel_max = -0.07, 0.07

# مقداردهی اولیه وزن‌ها
weights = np.zeros((num_tilings, tiles_per_dim, tiles_per_dim, num_actions))

# تابع تبدیل حالت پیوسته به کاشی‌های فعال
def get_active_tiles(position, velocity):
    pos_scale = tiles_per_dim / (pos_max - pos_min)
    vel_scale = tiles_per_dim / (vel_max - vel_min)

    active_tiles = []
    for i in range(num_tilings):
        # ایجاد آفست برای هر لایه (Tiling)
        offset = i / num_tilings
        p_idx = int((position - pos_min) * pos_scale + offset)
        v_idx = int((velocity - vel_min) * vel_scale + offset)

        # محدود کردن ایندکس‌ها برای جلوگیری از خطا
        p_idx = max(0, min(p_idx, tiles_per_dim - 1))
        v_idx = max(0, min(v_idx, tiles_per_dim - 1))
        active_tiles.append((i, p_idx, v_idx))
    return active_tiles

def get_q_value(active_tiles, action):
    q_val = 0
    for tiling, p, v in active_tiles:
        q_val += weights[tiling, p, v, action]
    return q_val

def choose_action(active_tiles, epsilon):
    if np.random.rand() < epsilon:
        return np.random.choice(num_actions)
    q_values = [get_q_value(active_tiles, a) for a in range(num_actions)]
    return np.argmax(q_values)

# پارامترهای آموزش
alpha = 0.1 / num_tilings
gamma = 1.0
epsilon = 0.1
num_episodes = 500  # برای بخش گرافیکی ۵۰۰ اپیزود کافیست تا یاد بگیرد

# ایجاد محیط با قابلیت رندر گرافیکی
# نکته: در ابتدا بدون رندر آموزش می‌دهیم تا سریع تمام شود، سپس رندر را فعال می‌کنیم
env = gym.make("MountainCar-v0")

print("شروع آموزش در محیط Gymnasium...")

for episode in range(num_episodes):
    # فعال سازی رندر در ۱۰ اپیزود آخر برای مشاهده نتیجه
    if episode > num_episodes - 10:
        env.close()
        env = gym.make("MountainCar-v0", render_mode="human")

    obs, _ = env.reset()
    active_tiles = get_active_tiles(obs[0], obs[1])
    action = choose_action(active_tiles, epsilon)

    step_count = 0
    while True:
        next_obs, reward, terminated, truncated, _ = env.step(action)
        next_active_tiles = get_active_tiles(next_obs[0], next_obs[1])

        target = reward
        if not terminated:
            next_action = choose_action(next_active_tiles, epsilon)
            target += gamma * get_q_value(next_active_tiles, next_action)

        # آپدیت وزن‌ها (Semi-gradient Sarsa Update)
        q_current = get_q_value(active_tiles, action)
        delta = target - q_current
        for tiling, p, v in active_tiles:
            weights[tiling, p, v, action] += alpha * delta

        if terminated or truncated:
            break

        active_tiles = next_active_tiles
        action = next_action
        step_count += 1

    if (episode + 1) % 50 == 0:
        print(f"Episode {episode + 1}: Steps = {step_count}")

env.close()
print("آموزش به پایان رسید.")
